# Day 13: Dataclasses, topological sort, nn.Module

*Week 3: PyTorch fundamentals and dynamic programming*

**Time budget:** about 60–90 minutes. **A** is Python fluency (~15 min), **B** is an interview-style algorithm (~25 min), **C** is NumPy / PyTorch / ML (~30 min).

**For each problem:**
1. Plan on paper or in comments before you type any code.
2. Work on it for 20 minutes before asking for a hint.
3. Run the test cell under it. Don't edit the tests.
4. Answer the **Say out loud** prompts, out loud. Interviews grade the explanation as much as the code.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

## A. Config with a dataclass

`Python`

Write a `@dataclass` called `TrainConfig` with fields and defaults `lr=1e-3`, `batch_size=64`, `epochs=10`, `z_dim=32`. In `__post_init__`, raise `ValueError` if `lr <= 0` or `batch_size <= 0`.

A dataclass writes `__init__`, `__repr__` and `__eq__` for you. Compare with the `Vector2D` you wrote by hand on Day 5.

**Look up:** `from dataclasses import dataclass, asdict`, type annotations on fields, `__post_init__`.

In [ ]:
# your code here

In [ ]:
# Tests: run after your solution. Don't edit.
from dataclasses import asdict
cfg = TrainConfig()
assert (cfg.lr, cfg.batch_size, cfg.epochs, cfg.z_dim) == (1e-3, 64, 10, 32)
assert TrainConfig(lr=0.01, epochs=5).epochs == 5
assert asdict(TrainConfig()) == {"lr": 1e-3, "batch_size": 64, "epochs": 10, "z_dim": 32}
assert TrainConfig() == TrainConfig(), "dataclasses get __eq__ for free"
assert repr(cfg).startswith("TrainConfig(")
for bad in ({"lr": 0}, {"batch_size": -1}):
    try:
        TrainConfig(**bad); raise AssertionError(f"{bad} should raise ValueError")
    except ValueError:
        pass
print("✅ A passed")

## B. Course schedule (topological sort)

`Algorithms`

There are `n` courses, numbered 0 to n−1. `prereqs` is a list of pairs `[a, b]` meaning "you must take `b` before `a`".

1. `course_order(n, prereqs)`: return any valid order to take all courses, or `[]` if it's impossible (a cycle).
2. `can_finish(n, prereqs)`: `True` or `False`. One line, using (1).

Use **Kahn's algorithm**: count each course's incoming edges (its "in-degree"), start a queue with every course that has none, and each time you take a course, decrement the in-degree of the courses that depend on it.

**Say out loud:** how does this detect a cycle? Where does topological sort show up in ML? (Hint: autograd.)

In [ ]:
def course_order(n, prereqs):
    # your code here
    pass


def can_finish(n, prereqs):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
def valid(order, n, prereqs):
    pos = {c: i for i, c in enumerate(order)}
    return sorted(order) == list(range(n)) and all(pos[b] < pos[a] for a, b in prereqs)
cases = [(2, [[1, 0]]), (4, [[1, 0], [2, 0], [3, 1], [3, 2]]), (3, []), (1, [])]
for n, pre in cases:
    assert valid(course_order(n, pre), n, pre), (n, pre)
    assert can_finish(n, pre) is True
assert course_order(2, [[1, 0], [0, 1]]) == [] and can_finish(2, [[1, 0], [0, 1]]) is False
assert can_finish(3, [[0, 1], [1, 2], [2, 0]]) is False
print("✅ B passed")

## C. An MLP as an nn.Module

`PyTorch`

Write `class MLP(nn.Module)` with `__init__(self, in_dim, hidden_dim, out_dim, n_hidden_layers=2)` and `forward(self, x)`.

Structure, for `n_hidden_layers=2`:
`Linear(in, hidden) → ReLU → Linear(hidden, hidden) → ReLU → Linear(hidden, out)`.
There is **no activation after the last layer**. Explain to yourself why: for classification the outputs are logits, and `cross_entropy` applies log-softmax itself.

**Look up:** `super().__init__()` (forgetting it is the classic error), `nn.Sequential`, building layers in a list and unpacking it with `*`, `nn.ReLU`.

**Say out loud:** the parameter count formula for this network.

In [ ]:
class MLP(nn.Module):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
torch.manual_seed(0)
for n_hidden in [1, 2, 3]:
    m = MLP(10, 32, 3, n_hidden_layers=n_hidden)
    assert m(torch.randn(5, 10)).shape == (5, 3)
    linears = [mod for mod in m.modules() if isinstance(mod, nn.Linear)]
    relus = [mod for mod in m.modules() if isinstance(mod, nn.ReLU)]
    assert len(linears) == n_hidden + 1 and len(relus) == n_hidden, n_hidden
m = MLP(10, 32, 3, n_hidden_layers=2)
n_params = sum(p.numel() for p in m.parameters())
assert n_params == (10 * 32 + 32) + (32 * 32 + 32) + (32 * 3 + 3), n_params
assert (m(torch.randn(100, 10)) < 0).any(), "no activation after the last layer"
print("✅ C passed")

## Wrap-up (5 minutes)

- For each problem: what's its time and space complexity? Which edge cases did you handle?
- Anything that took more than 20 minutes, or needed a hint: add it to the **Redo list** in `README.md` and solve it again from scratch in 3 days, without looking at today's code.
- Tick off today in `README.md`.